# Management and scheme predictors: overlap and identity

This focused notebook completes the initial **related categorical** review of
`scheme_management`, `scheme_name`, `management`, `management_group`. It describes the supplied training and test predictors,
then uses the labelled training rows to identify relationships worth
carrying into a leakage-safe modelling pipeline.

The audit separates low-cardinality management types from the high-cardinality scheme identifier.

This is exploratory evidence, not fitted preprocessing. Category pooling,
imputation, encoding and scaling must be learned inside each training fold.


## Consistent audit contract

Every focused predictor audit answers the same questions before adding
type-specific checks:

1. What is explicitly missing, and what looks like a sentinel?
2. What range or category coverage is present in training and test?
3. How much of the test set is exposed to unseen training levels?
4. Does the labelled distribution vary enough to justify retaining the field?
5. What exact baseline treatment follows from the evidence?

Target-rate tables flag support rather than treating tiny groups as reliable.
Train/test comparisons are descriptive and do not use the hidden test labels.


In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

source_directory = str(Path("../src").resolve())
if source_directory not in sys.path:
    sys.path.insert(0, source_directory)

from predictor_audit import (
    MISSING_CATEGORY,
    analysis_categories,
    categorical_summary,
    categorical_target_profile,
    category_frequency_table,
    cramer_v,
    hierarchy_conflicts,
    hierarchy_summary,
    numeric_summary,
    numeric_target_summary,
    normalise_categories,
    sentinel_mask,
    source_blank_mask,
    text_normalisation_summary,
)
from source_data_validation import (
    validate_aligned_ids,
    validate_label_frame,
    validate_raw_feature_schema,
)

data_directory = Path("../data")
training_features = pd.read_csv(
    data_directory / "TrainingSetValues.csv",
    keep_default_na=False,
)
training_labels = pd.read_csv(
    data_directory / "TrainingSetLabels.csv",
    keep_default_na=False,
)
test_features = pd.read_csv(
    data_directory / "TestSetValues.csv",
    keep_default_na=False,
)

validate_raw_feature_schema(training_features)
validate_raw_feature_schema(test_features)
validate_label_frame(training_labels)
validate_aligned_ids(training_features, training_labels)

training_data = training_features.merge(
    training_labels,
    on="id",
    validate="one_to_one",
)
audited_features = ['scheme_management', 'scheme_name', 'management', 'management_group']
assert set(audited_features).issubset(training_features.columns)

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 80)
print(
    f"Validated {len(training_features):,} training rows and "
    f"{len(test_features):,} test rows for {len(audited_features)} predictors."
)


Validated 59,400 training rows and 14,850 test rows for 4 predictors.


## 1. Missingness, cardinality and test coverage

Source blanks, pandas nulls and configured sentinel strings are reported separately.
Semantic sentinels such as `unknown` stay visible in frequency and target tables;
they are not silently merged with blank values.
Rare means fewer than 50 training rows; it is a diagnostic threshold, not
a preprocessing choice. Total-variation distance compares marginal shares.


In [2]:
category_overview = categorical_summary(
    training_features,
    test_features,
    audited_features,
    rare_threshold=50,
    sentinel_tokens_by_column={},
)
display(category_overview)


,training explicit missing,training source blank rows,training sentinel rows,test explicit missing,test source blank rows,test sentinel rows,training levels,test levels,training levels with <50 rows,training rows in rare levels (%),test-only levels,test rows in unseen levels (%),training-only levels,marginal total-variation distance
feature,,,,,,,,,,,,,,
scheme_management,0,3877,1,0,969,0,12,11,1,0.00,0,0.00,1,0.0095
scheme_name,0,28166,677,0,7092,161,2526,1712,2435,32.97,137,1.02,951,0.1076
management,0,0,561,0,0,122,12,12,0,0.00,0,0.00,0,0.0082
management_group,0,0,561,0,0,122,5,5,0,0.00,0,0.00,0,0.0062


## 2. Most common values


In [3]:
for feature in audited_features:
    print()
    print(feature)
    display(category_frequency_table(training_features, test_features, feature, top_n=10))



scheme_management


,training rows,training (%),test rows,test (%)
scheme_management,,,,
vwc,36793,61.94,9124,61.44
wug,5206,8.76,1290,8.69
<missing/blank>,3877,6.53,969,6.53
water authority,3153,5.31,822,5.54
wua,2883,4.85,668,4.5
water board,2748,4.63,714,4.81
parastatal,1680,2.83,444,2.99
private operator,1063,1.79,263,1.77
company,1061,1.79,280,1.89



scheme_name


,training rows,training (%),test rows,test (%)
scheme_name,,,,
<missing/blank>,28166,47.42,7092,47.76
k,685,1.15,176,1.19
none,669,1.13,159,1.07
borehole,546,0.92,158,1.06
chalinze wate,406,0.68,96,0.65
m,400,0.67,90,0.61
danida,379,0.64,104,0.7
government,320,0.54,75,0.51
bagamoyo wate,296,0.5,88,0.59



management


,training rows,training (%),test rows,test (%)
management,,,,
vwc,40507,68.19,10117,68.13
wug,6515,10.97,1593,10.73
water board,2933,4.94,755,5.08
wua,2535,4.27,583,3.93
private operator,1971,3.32,533,3.59
parastatal,1768,2.98,461,3.1
water authority,904,1.52,219,1.47
other,844,1.42,239,1.61
company,685,1.15,174,1.17



management_group


,training rows,training (%),test rows,test (%)
management_group,,,,
user-group,52490,88.37,13048,87.87
commercial,3638,6.12,953,6.42
parastatal,1768,2.98,461,3.1
other,943,1.59,266,1.79
unknown,561,0.94,122,0.82


## 3. Relationship with `status_group`

The tables display the most supported levels first and mark whether each
level has at least 150 training rows. Small groups are leads
for later validation, not stable target encodings.


In [4]:
for feature in audited_features:
    print()
    print(feature)
    profile = categorical_target_profile(
        training_data,
        feature,
        minimum_support=150,
    )
    display(profile.head(15))



scheme_management


status_group,rows,meets support threshold,functional (%),functional needs repair (%),non functional (%)
scheme_management,,,,,
vwc,36793,True,51.53,6.34,42.12
wug,5206,True,57.74,12.91,29.35
<missing/blank>,3877,True,48.31,5.75,45.94
water authority,3153,True,51.32,14.21,34.48
wua,2883,True,69.20,8.29,22.51
water board,2748,True,74.71,4.04,21.25
parastatal,1680,True,57.50,12.02,30.48
private operator,1063,True,68.58,2.16,29.26
company,1061,True,50.33,3.49,46.18



scheme_name


status_group,rows,meets support threshold,functional (%),functional needs repair (%),non functional (%)
scheme_name,,,,,
<missing/blank>,28166,True,51.44,7.11,41.45
k,685,True,55.04,16.79,28.18
none,669,True,63.68,4.48,31.84
borehole,546,True,37.36,4.76,57.88
chalinze wate,406,True,85.96,0.00,14.04
m,400,True,49.25,14.00,36.75
danida,379,True,52.51,4.49,43.01
government,320,True,46.88,10.00,43.12
bagamoyo wate,296,True,70.27,0.00,29.73



management


status_group,rows,meets support threshold,functional (%),functional needs repair (%),non functional (%)
management,,,,,
vwc,40507,True,50.42,6.89,42.69
wug,6515,True,59.95,9.90,30.15
water board,2933,True,73.99,9.04,16.98
wua,2535,True,69.07,8.09,22.84
private operator,1971,True,74.89,2.23,22.88
parastatal,1768,True,57.69,11.93,30.37
water authority,904,True,49.34,5.75,44.91
other,844,True,59.83,6.52,33.65
company,685,True,38.98,2.19,58.83



management_group


status_group,rows,meets support threshold,functional (%),functional needs repair (%),non functional (%)
management_group,,,,,
user-group,52490,True,53.82,7.44,38.73
commercial,3638,True,61.43,3.22,35.35
parastatal,1768,True,57.69,11.93,30.37
other,943,True,55.99,5.94,38.07
unknown,561,True,39.93,4.81,55.26


## 4. Related-field consistency

A deterministic child-to-parent mapping makes the parent derivable from
the child in this dataset. That is redundancy evidence, not automatic
permission to discard the child: granularity, unseen levels and model
behaviour still determine which representation is safer.


In [5]:
hierarchy_relationships = [('management', 'management_group'), ('scheme_name', 'scheme_management')]
display(
    hierarchy_summary(
        training_features,
        test_features,
        hierarchy_relationships,
    )
)
for child, parent in hierarchy_relationships:
    conflicts = hierarchy_conflicts(training_features, child, parent)
    if not conflicts.empty:
        print()
        print(f"Training conflicts for {child} -> {parent}")
        display(conflicts)


complete rows  child levels  \
relationship                     frame                                   
management -> management_group   training          59400            12   
                                 test              14850            12   
scheme_name -> scheme_management training          30928          2508   
                                 test               7701          1702   

                                           parent levels  \
relationship                     frame                     
management -> management_group   training              5   
                                 test                  5   
scheme_name -> scheme_management training             10   
                                 test                 10   

                                           ambiguous child levels  \
relationship                     frame                              
management -> management_group   training                       0   
                                 test                           0   
scheme_name -> scheme_management training                     300   
                                 test                         137   

                                           rows in ambiguous child levels  \
relationship                     frame                                      
management -> management_group   training                               0   
                                 test                                   0   
scheme_name -> scheme_management training                           11299   
                                 test                                2069   

                                           deterministic child-to-parent  \
relationship                     frame                                     
management -> management_group   training                           True   
                                 test                               True   
scheme_name -> scheme_management training                          False   
                                 test                              False   

                                           one-to-one level mapping  
relationship                     frame                               
management -> management_group   training                     False  
                                 test                         False  
scheme_name -> scheme_management training                     False  
                                 test                         False


Training conflicts for scheme_name -> scheme_management


,rows,parent levels,parents
child,,,
<missing/blank>,28166,13,"<missing/blank>, company, none, other, parastatal, private operator, swc, tr..."
k,685,6,"<missing/blank>, parastatal, vwc, water authority, wua, wug"
none,669,8,"company, other, parastatal, private operator, vwc, water authority, wua, wug"
borehole,546,9,"<missing/blank>, company, other, parastatal, private operator, trust, vwc, w..."
chalinze wate,406,2,"vwc, wua"
m,400,7,"company, parastatal, private operator, vwc, water authority, wua, wug"
danida,379,2,"<missing/blank>, vwc"
government,320,6,"<missing/blank>, parastatal, private operator, vwc, water authority, wug"
bagamoyo wate,296,4,"company, parastatal, private operator, vwc"


## 5. Preserve source blanks and literal scheme-name sentinels

Loading with `keep_default_na=False` prevents pandas from merging the literal
string `None` with an empty CSV cell. The states below remain separate because
their target distributions differ and only the blank is structurally absent.


In [6]:
def scheme_source_state(series):
    raw = series.astype("string")
    return pd.Series(
        np.select(
            [
                raw.str.strip().eq(""),
                raw.eq("None"),
                raw.eq("none"),
                raw.str.strip().str.casefold().eq("no scheme"),
            ],
            ["blank", "literal None", "literal none", "no scheme"],
            default="other recorded name",
        ),
        index=series.index,
    )

for feature in ["scheme_management", "scheme_name"]:
    source_state = scheme_source_state(training_data[feature])
    source_profile = pd.crosstab(
        source_state,
        training_data["status_group"],
        normalize="index",
    ).mul(100).round(2)
    source_profile.insert(0, "rows", source_state.value_counts())
    print()
    print(feature)
    display(source_profile)



scheme_management


status_group,rows,functional,functional needs repair,non functional
row_0,,,,
blank,3877,48.31,5.75,45.94
literal None,1,100.00,0.00,0.00
other recorded name,55522,54.73,7.37,37.90



scheme_name


status_group,rows,functional,functional needs repair,non functional
row_0,,,,
blank,28166,51.44,7.11,41.45
literal None,644,63.98,4.66,31.37
literal none,25,56.00,0.00,44.00
no scheme,37,72.97,2.70,24.32
other recorded name,30528,56.73,7.48,35.80


## 6. Management-field overlap


In [7]:
scheme = analysis_categories(training_features["scheme_management"])
management = analysis_categories(training_features["management"])
overlap_table = pd.crosstab(scheme, management)
display(overlap_table)
display(pd.DataFrame({
    "Cramer's V": [cramer_v(overlap_table)],
    "exact normalised matches (%)": [scheme.eq(management).mean() * 100],
}, index=["scheme_management vs management"]).round(3))


management,company,other,other - school,parastatal,private operator,trust,unknown,vwc,water authority,water board,wua,wug
scheme_management,,,,,,,,,,,,
<missing/blank>,1,186,0,11,119,1,468,2450,2,0,6,633
company,674,2,0,25,224,0,0,135,0,0,0,1
none,0,0,0,0,0,0,0,1,0,0,0,0
other,0,519,0,1,64,1,8,41,19,0,0,113
parastatal,0,0,0,1568,59,0,4,47,1,1,0,0
private operator,2,4,0,3,963,1,0,83,5,1,0,1
swc,0,0,87,0,0,0,0,9,0,0,0,1
trust,1,0,0,1,1,61,0,6,1,1,0,0
vwc,2,49,12,123,160,2,76,35388,8,41,109,823


,Cramer's V,exact normalised matches (%)
scheme_management vs management,0.749,83.057


## Training/test handoff


In [8]:
display(
    category_overview[[
        "training levels",
        "test levels",
        "test-only levels",
        "test rows in unseen levels (%)",
        "training rows in rare levels (%)",
        "marginal total-variation distance",
    ]].sort_values("test rows in unseen levels (%)", ascending=False)
)


,training levels,test levels,test-only levels,test rows in unseen levels (%),training rows in rare levels (%),marginal total-variation distance
feature,,,,,,
scheme_name,2526,1712,137,1.02,32.97,0.1076
scheme_management,12,11,0,0.00,0.00,0.0095
management,12,12,0,0.00,0.00,0.0082
management_group,5,5,0,0.00,0.00,0.0062


## Decision register

The register separates observed evidence from the proposed baseline action.
A retained field is still a candidate: later validation must show whether it
improves generalisation and whether a coarser related representation is safer.


In [9]:
decision_register = pd.DataFrame([{'feature': 'scheme_management', 'quality finding': '3,877 training rows are blank; it overlaps management but is not equivalent.', 'baseline treatment': 'Retain unknown as a level; compare against management by ablation.', 'risk to verify': 'Parallel management fields may add redundancy more than signal.'}, {'feature': 'scheme_name', 'quality finding': '28,166 rows are blank, 644 literal None; 1.26% of test rows use unseen names.', 'baseline treatment': 'Omit raw one-hot form initially; preserve blank/None and test fold-fitted pooling later.', 'risk to verify': 'Scheme identity can memorise local geography.'}, {'feature': 'management', 'quality finding': 'Twelve complete levels map deterministically to five management_group levels.', 'baseline treatment': 'Use as the initial granular management representation.', 'risk to verify': 'Some sparse levels may need grouping.'}, {'feature': 'management_group', 'quality finding': 'Deterministic coarse parent retains much less descriptive target association.', 'baseline treatment': 'Treat as likely redundant; compare coarse versus granular representation.', 'risk to verify': 'Coarser levels may generalise better to rare management methods.'}])
display(decision_register.set_index("feature"))


,quality finding,baseline treatment,risk to verify
feature,,,
scheme_management,"3,877 training rows are blank; it overlaps management but is not equivalent.",Retain unknown as a level; compare against management by ablation.,Parallel management fields may add redundancy more than signal.
scheme_name,"28,166 rows are blank, 644 literal None; 1.26% of test rows use unseen names.",Omit raw one-hot form initially; preserve blank/None and test fold-fitted po...,Scheme identity can memorise local geography.
management,Twelve complete levels map deterministically to five management_group levels.,Use as the initial granular management representation.,Some sparse levels may need grouping.
management_group,Deterministic coarse parent retains much less descriptive target association.,Treat as likely redundant; compare coarse versus granular representation.,Coarser levels may generalise better to rare management methods.


### Handoff to modelling

Prefer one validated representation per management concept; treat scheme identity as a high-cardinality feature with an explicit omission baseline.

- Preserve raw source frames and implement the stated sentinel rules on copies.
- Fit imputers, rare-level grouping and encoders on each training fold only.
- Map unseen validation or test categories to an explicit fallback.
- Compare the stated baseline treatment with a simple omission ablation.
- Revisit target-rate observations after the reproducible stratified split exists.
